# AMOS MRI · 증강 OFF · Train 50 / Val 10 · 250 epochs · deep supervision ON

서버에서 **필요 시 MRI 선택 다운로드 → 전처리 → GPU smoke test → OFF 학습·평가**를 tmux로 실행합니다. **학습 50개 / 검증 10개**를 사용자 지정 fold 0 하나로 고정합니다. 공식 학습 40개를 유지하고, 공식 검증 20개에서 SPLIT_SEED로 고정 추출한 10개를 학습으로 옮깁니다. 공식 40/20 분할 결과와 직접 비교할 수 없습니다. 5-fold 교차검증 결과가 아닙니다. 시험 40개는 사용하지 않습니다.

모델은 일반 3D U-Net 기반 nnU-Net `3d_fullres`, 사전학습 없이 초기화합니다. 모든 무작위 영상 변형을 OFF로 설정한 실험 하나만 실행합니다. OFF에서도 patch sampling, foreground oversampling, crop/padding, 정규화, deep supervision 정답 변환은 유지합니다. 추론 mirror TTA는 기존 설정을 유지합니다.

**AMP=False가 기본입니다.** FP32 학습·검증·예측, TF32 OFF입니다. `AMP=True`는 CUDA FP16 autocast + GradScaler를 사용합니다. 입력·파라미터는 FP32여도 AMP는 켜져 있을 수 있습니다. 선택한 정밀도를 적용하고, 추론 누적은 FP32 CPU로 수행합니다.

250 train steps / 50 validation steps per epoch, SGD lr=.01 / momentum=.99 / Nesterov / weight decay=3e-5, 250-epoch PolyLR, 기본 Dice+CE와 deep supervision을 사용합니다. 공통 seed는 초기화를 맞추지만 비동기 augmentation 때문에 비트 단위 재현을 보장하지 않습니다.

**이 컴퓨터에서는 실행하지 마세요. Linux GPU 서버 전용입니다.** 로컬의 `amos-data` 전체와 최신 repo 전체를 서버에 나란히 복사하면 기존 786 MB MRI를 재사용합니다. 복사하지 않으면 원본 ZIP의 필요한 부분만 다운로드합니다. 전체 ZIP, CT, 시험 영상은 저장하지 않습니다.

[공식 데이터](https://zenodo.org/records/7262581) · [PyTorch AMP](https://docs.pytorch.org/docs/2.8/amp.html)

## 1. 경로와 조건
Python 3.11 venv의 Jupyter 커널을 선택하세요. 다른 실험이 실행 중인 환경에 패키지를 설치하지 마세요. 별도 venv 사용을 권장합니다. `git`, `tmux`, NVIDIA 드라이버는 서버에 설치되어 있어야 합니다.

`PLANNER_GPU_GB=8`은 기본 일반 U-Net 계획의 메모리 예산입니다. **실제 FP32 사용량 8 GB를 보장하지 않습니다.** batch 2부터 시험하고, OOM이면 AMP를 켜서 새 실행으로 시작하세요. patch/batch/spacing은 기존 계획을 유지합니다. 전처리 프로세스 2개로 RAM 사용을 제한합니다.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, shlex, shutil, importlib.util
sys.dont_write_bytecode = True
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'nnunetv2').is_dir()), None)
if REPO is None:
    raise RuntimeError('nnUNet repo 안에서 노트북을 여세요.')
EXPERIMENT = REPO/'experiments/amos'
STORAGE_ROOT = (REPO.parent/'amos-data').resolve()
AMP = False                 # OOM이면 True로 바꾸고 이 셀부터 다시 실행
RUN_LABEL = 'v1'             # 같은 정밀도로 새 실험은 v2 등 새 이름 사용
GPU = '0'
BATCH_SIZE = 2
PLANNER_GPU_GB = 8
WORKERS = 2
SEED = 12345
SPLIT_SEED = 12345           # 분할 고정. 변경 시 RUN_LABEL도 변경하세요.
RUN_ID = f"amos-mri-noaug-tr50val10-s{SPLIT_SEED}-250-{'amp' if AMP else 'fp32'}-{RUN_LABEL}"
EVALUATE_BEST = True
INSTALL_PACKAGES = False
START_SUITE = False
RESUME = False               # 같은 설정의 중단 실행만 True. 정밀도 변경 재개 금지.
required = [EXPERIMENT/'scripts/workflow.py', EXPERIMENT/'scripts/download_mri.py',
            REPO/'nnunetv2/training/nnUNetTrainer/variants/data_augmentation/nnUNetTrainerAMOS250.py']
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('최신 repo 전체를 복사하세요: '+str(missing))
print('Kernel:', sys.executable)
print('Data:', STORAGE_ROOT, '| Run:', RUN_ID, '| AMP:', AMP)

In [ ]:
if INSTALL_PACKAGES:
    if sys.platform != 'linux' or sys.prefix == sys.base_prefix or sys.version_info[:2] != (3,11):
        raise RuntimeError('Linux 서버의 Python 3.11 venv에서 설치하세요.')
    if any(m in sys.modules for m in ('torch','numpy','nnunetv2')):
        raise RuntimeError('커널을 재시작한 뒤 설치 셀을 실행하세요.')
    from importlib.metadata import version, PackageNotFoundError
    specs = ['torch==2.8.0+cu128','torchvision==0.23.0+cu128']
    try:
        version('torchaudio')
    except PackageNotFoundError:
        pass
    else:
        specs.append('torchaudio==2.8.0+cu128')
    pip = [sys.executable,'-m','pip']
    subprocess.check_call(pip+['install','--no-cache-dir',*specs,'--index-url','https://download.pytorch.org/whl/cu128'])
    subprocess.check_call(pip+['install','--no-cache-dir',*specs,'-e',str(REPO),
        'batchgeneratorsv2==0.3.8','dynamic-network-architectures==0.4.4',
        'numpy>=1.24,<3','pandas>=2,<3','matplotlib>=3.8,<4','ipykernel>=6,<7'])
    subprocess.check_call(pip+['check'])
    subprocess.check_call([sys.executable,'-c',
        "import torch; print(torch.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"],
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=GPU))
    print('설치 완료. 커널 재시작 후 INSTALL_PACKAGES=False로 바꾸세요.')
else:
    print('설치 생략')

## 2. 백그라운드 실행
`START_SUITE=True`로 바꾸고 실행하세요. 전처리도 tmux 안에서 진행되므로 브라우저를 닫아도 이어집니다. 서버 자체가 종료되면 중단됩니다.

원본은 hard link로 연결하여 중복 저장하지 않습니다. Dataset620은 AMOS MRI 전용으로 사용하세요. 전처리 후 실사용 디스크와 patch/spacing을 로그에 출력합니다. 10 GiB 여유 공간 검사는 최소 검사일 뿐, 최종 용량 보장이 아닙니다.

정밀도 전환 시 RUN_ID가 자동으로 달라집니다. **FP32에서 OOM이 발생하면 AMP=True, RESUME=False로 다시 시작**하세요. AMP 실험은 전처리만 재사용하고 증강 OFF 모델 하나를 새로 학습합니다. 실패한 실험 파일은 자동 삭제하지 않습니다. 같은 실행을 이어가는 경우만 RESUME=True입니다.

**기존 40/20 실행을 먼저 중단한 뒤 새 RUN_ID, RESUME=False로 시작하세요.** 분할이 바뀌므로 기존 checkpoint를 이어 쓰지 않습니다. 기존 전처리가 완료되어 있으면 데이터를 hard link로 재사용하고, 새 분할은 별도 `nnUNet_preprocessed_train50val10_seed12345` 아래에 저장합니다. 기존 분할·결과는 보존됩니다. 전처리가 없으면 서버에서 새로 수행합니다.


In [ ]:
spec = importlib.util.spec_from_file_location('amos_workflow', EXPERIMENT/'scripts/workflow.py')
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)
cfg = dict(storage_root=str(STORAGE_ROOT),run_id=RUN_ID,gpu=GPU,amp=AMP,
           batch_size=BATCH_SIZE,planner_gb=PLANNER_GPU_GB,workers=WORKERS,seed=SEED,
           evaluate_best=EVALUATE_BEST, split_mode='train50_val10', split_seed=SPLIT_SEED)
workflow.validate(cfg)
p = workflow.locations(cfg)
SESSION = 'amos-'+RUN_ID
CONFIG_PATH = p['suite']/'config.json'
LOG_PATH = p['suite']/'runner.log'
if START_SUITE:
    if sys.platform != 'linux':
        raise RuntimeError('서버에서만 실행하세요.')
    for tool in ('git','tmux','nvidia-smi'):
        if shutil.which(tool) is None:
            raise RuntimeError(f'서버에 먼저 설치하세요: {tool}')
    if subprocess.run(['tmux','has-session','-t',SESSION],capture_output=True).returncode == 0:
        print('이미 실행 중:', SESSION)
    else:
        workflow.frozen(CONFIG_PATH,cfg)
        command = [sys.executable,'-u','-B',str(EXPERIMENT/'scripts/workflow.py'),'--config',str(CONFIG_PATH)]
        if RESUME:
            command.append('--resume')
        shell_command = 'exec '+shlex.join(command)+' >> '+shlex.quote(str(LOG_PATH))+' 2>&1'
        subprocess.check_call(['tmux','new-session','-d','-s',SESSION,shell_command],cwd=REPO)
        print('시작:',SESSION,'| log:',LOG_PATH)
else:
    print('실행 안 함. START_SUITE=True로 설정하세요.')
print('결과:',p['suite'])

## 3. 진행 상태와 로그
이 셀은 여러 번 실행할 수 있습니다. `completed`가 되면 증강 OFF 실험이 완료된 것입니다. OOM으로 실패하면 로그를 확인한 후 AMP를 변경하세요. smoke test는 실제 patch 3개 배치의 역전파까지 확인하지만, 모든 영상에서의 메모리 상한을 보장하지는 않습니다.

In [ ]:
status = p['suite']/'status.json'
print(json.dumps(workflow.read(status),indent=2,ensure_ascii=False) if status.exists() else '아직 상태 없음')
if LOG_PATH.exists():
    from collections import deque
    with LOG_PATH.open(errors='replace') as f:
        print(''.join(deque(f,maxlen=35)))
for theme in workflow.themes(cfg):
    probe = p['suite']/f'smoke-{theme}.json'
    if probe.exists():
        print(theme, workflow.read(probe))
if STORAGE_ROOT.exists():
    print('디스크 여유 GiB:',round(shutil.disk_usage(STORAGE_ROOT).free/2**30,2))

## 4. Train loss / validation loss
epoch별 loss는 무작위 patch 기준이며, 전체 검증 영상 Dice와 다릅니다. DS가 켜진 Dice+CE 손실은 음수가 될 수 있습니다. 아래 그래프는 1부터 시작하는 epoch 번호를 사용합니다.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
fig,ax = plt.subplots(figsize=(8,4))
for theme in workflow.themes(cfg):
    rows={}
    folder=workflow.model_dir(cfg,theme)/'fold_0'
    for logfile in sorted(folder.glob('training_log_*.txt'),key=lambda f:f.stat().st_mtime_ns):
        epoch=None
        for line in logfile.read_text(errors='replace').splitlines():
            match=re.search(r'\bEpoch\s+(\d+)\s*$',line)
            if match:
                epoch=int(match[1])+1;rows[epoch]={'epoch':epoch}
            match=re.search(r'\b(train_loss|val_loss)\s+([-+\deE.]+)',line)
            if match and epoch is not None:
                rows[epoch][match[1]]=float(match[2])
    frame=pd.DataFrame([rows[e] for e in sorted(rows) if 'train_loss' in rows[e] and 'val_loss' in rows[e]])
    if not frame.empty:
        ax.plot(frame.epoch,frame.train_loss,label='Train loss')
        ax.plot(frame.epoch,frame.val_loss,label='Validation loss')
        ax.legend()
    ax.set(title=f'Augmentation {theme.upper()}',xlabel='Epoch',ylabel='Loss')
    ax.grid(alpha=.2)
plt.tight_layout()
if p['suite'].exists():
    fig.savefig(p['suite']/'loss-curves.png',dpi=150)
plt.show()

## 5. 전체 검증 영상 성능 비교
**증강 OFF 모델의 final과 best 결과**를 표시합니다. best는 nnU-Net EMA pseudo-Dice로 선택하며, 같은 검증 세트로 모델 선택과 평가를 하므로 독립 시험 결과가 아닙니다. 검증 10개는 기존 공식 검증 세트에서 선택되어 장비 도메인이 제한되어 있으며 시험 OOD 성능을 의미하지 않습니다.

Dice/IoU는 장기별 case 평균 후 전경 15개 장기 평균입니다. 결측 장기는 nnU-Net 기본 NaN 처리 방식입니다. 예측 마스크는 각 모델의 `fold_0/validation`과 `validation_best`에 따로 저장합니다. 확률 NPZ는 저장하지 않습니다.

In [ ]:
path=p['suite']/'comparison.json'
if path.exists():
    table=pd.DataFrame(workflow.read(path))
    display(table)
    if not table.empty:
        ax=table.pivot(index='augmentation',columns='checkpoint',values='Dice').plot.bar(ylim=(0,1),ylabel='Mean foreground Dice')
        plt.tight_layout()
        plt.savefig(p['suite']/'comparison.png',dpi=150)
        plt.show()
    organs=workflow.read(p['raw']/'dataset.json')['labels']
    names={str(v):k for k,v in organs.items()}
    details=[]
    for theme in workflow.themes(cfg):
        for checkpoint,subdir in [('final','validation'),('best','validation_best')]:
            summary=workflow.model_dir(cfg,theme)/'fold_0'/subdir/'summary.json'
            if summary.exists():
                for label,metrics in workflow.read(summary)['mean'].items():
                    details.append(dict(augmentation=theme,checkpoint=checkpoint,organ=names.get(label,label),
                                        Dice=metrics['Dice'],IoU=metrics['IoU']))
    details=pd.DataFrame(details)
    display(details)
    details.to_csv(p['suite']/'per-organ-comparison.csv',index=False)
else:
    print('아직 평가 결과가 없습니다.')

## 6. 중단 / 재개
명시적으로 중단할 때만 아래 값을 True로 바꾸세요. tmux 세션을 종료합니다. 마지막 저장 checkpoint 이후의 진행은 유실될 수 있습니다. 같은 설정으로 재개하려면 중단 완료를 확인하고 RESUME=True로 시작 셀을 실행하세요. checkpoint가 없는 smoke/OOM 실패는 같은 설정에서 처음부터 다시 검사합니다.

GPU 검사나 학습 중 OOM이면 자동으로 AMP를 바꾸지 않습니다. AMP=True로 바꾸면 새 실험 이름을 사용하므로 기존 결과가 섞이지 않습니다. 전처리 계획이나 batch를 바꾸면 별도 전처리 계획이 생성되어 추가 디스크를 사용할 수 있습니다.

In [ ]:
STOP_SUITE = False
if STOP_SUITE:
    subprocess.run(['tmux','kill-session','-t',SESSION],check=False)
    print('중단 요청:',SESSION)